In [1]:
import ee
import os
import geemap
from dotenv import load_dotenv
load_dotenv()

# Initialize the Earth Engine module.
PROJECT_ID = os.getenv("PROJECT_ID")
ee.Initialize(project = os.getenv("PROJECT_ID"))

# Define Denmark Area
DENMARK = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filter(ee.Filter.eq('country_na', 'Denmark')).geometry()


In [2]:
# Calculate area of an image in km²
def dw_area(img):
    area_image = img.multiply(ee.Image.pixelArea())
    area = area_image.reduceRegion( 
        reducer = ee.Reducer.sum(),
        scale = 10,
        maxPixels = 1e13,
    )
    return ee.Number(area.get('label')).divide(pow(10,6))

# Calculate change matrix
# 9x9 matrix area classified per class vs area classified per class other year
def change_matrix(img, img_prev):
    for i in range(9):
        for j in range(9):
            dw_transition = img.eq(i).And(img_prev.eq(j))
            dw_transition_area = dw_area(dw_transition)
            display(dw_transition_area)


In [5]:
YEAR = 23

# DANISH AGRICULTURE AGENCY
dataset = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke" + str(YEAR))
dataset_prev = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke" + str(YEAR - 1))
# Transform data to raster format for lighter operations
daa_img = ee.Image.constant(0).paint(dataset, 1)
daa_img_prev = ee.Image.constant(0).paint(dataset_prev, 1)
# Image with are that changed from DAA (1 - new field, -1 - removed field)
daa_change_img = daa_img.subtract(daa_img_prev)
daa_change_img = daa_change_img.updateMask(daa_change_img.neq(0))


# DYNAMIC WORLD CLASSIFICATION OF DAA AREA
dw_img = ee.Image('projects/' + PROJECT_ID + '/assets/dw_20' + str(YEAR) + '_mode').clipToCollection(dataset)
dw_img_prev = ee.Image('projects/' + PROJECT_ID + '/assets/dw_20' + str(YEAR - 1) + '_mode').clipToCollection(dataset_prev)

## Change Matrix


In [6]:
change_matrix(dw_img, dw_img_prev)

#### New Fields Area

From the new field area, how did classification change? How was it classified before? How is it classified now?

New Field Area from 2024-2025: 90.54 km²

In [ ]:
change_matrix(dw_img.And(daa_change_img.eq(1)), dw_img_prev.And(daa_change_img.eq(1)))

#### Removed Fields Area

From the removed field area, how did classification change? How was it classified before? How is it classified now?

Removed Field Area from 2024-2025: 148.18 km²

In [ ]:
change_matrix(dw_img.And(daa_change_img.eq(-1)), dw_img_prev.And(daa_change_img.eq(-1)))

## Amount of Changes

In [ ]:
# Create an ImageCollection by using ee.Image.neq() for every consecutive year
# ee.ImageCollection.sum() returns a ee.Image with the number of changes per pixel
# 